# Did fuel-tax cuts get Europe driving again?
### A 3-hour taste of data-driven research

In early 2026 a war spiked oil prices across Europe. Sixteen governments rushed
out **fuel-tax cuts**. Did the cuts work? Did people change how they drove?

Today, we will study a policy question in a data-driven manner. In our case, that comes down to four habits, and
we'll build them in order:

1. **Define** the thing you measure — and notice it isn't quite the thing you want to measure.
2. **Trust** it — check if your effect is real before you believe it.
3. **Know its limits** — which questions can the data answer, which not?
4. **Constrain your claim** — not only significant effects are worth mentioning.

Along the way, watch for **biases**. It turns up in **three different disguises** today. Learn to tell them apart.

---
**How this notebook works**
- Run a cell with **Shift + Enter**. Most cells are already written — just run them.
- A few are yours to finish: marked **`# TODO`** with a hint. Replace the `...`.
- Cells marked **Go deeper** are optional — for when you're ahead and want to push.
- The workshop today is about **arguing**. The code is just how we get something
  true enough to argue about.

In [ ]:
# Run me first. Loads our tools and the workshop data.
#
# ┌─ INSTRUCTOR: for one-click Colab, paste a PUBLIC link to data.zip below. ──┐
# │ Works with a GitHub "raw" link OR a Google Drive share link.              │
# │ Leave it "" to use a local data/ folder, or to upload data.zip by hand.   │
DATA_URL = "https://raw.githubusercontent.com/Societal-Computing/fuel_impact_workshop/main/data.zip"
# └───────────────────────────────────────────────────────────────────────────┘

import io, json, re, pathlib, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

def _find_data():
    for p in [pathlib.Path("data"), pathlib.Path("workshop/data"), pathlib.Path(".")]:
        if (p / "busyness_daily.csv").exists():
            return p
    return None

def _download(url):
    if "drive.google.com" in url:                 # turn a Drive share link into a download
        m = re.search(r"/d/([\w-]+)", url) or re.search(r"[?&]id=([\w-]+)", url)
        file_id = m.group(1)
        try:
            import gdown                           # pre-installed on Colab
            gdown.download(id=file_id, output="data.zip", quiet=True)
            return pathlib.Path("data.zip").read_bytes()
        except Exception:
            url = f"https://drive.google.com/uc?export=download&id={file_id}"
    return requests.get(url, timeout=60).content

DATA = _find_data()
if DATA is None and DATA_URL:                      # one-click path: auto-download the data
    print("⬇️  Downloading the workshop data…")
    zipfile.ZipFile(io.BytesIO(_download(DATA_URL))).extractall(".")
    DATA = _find_data()
if DATA is None:                                   # fallback: manual upload on Colab
    try:
        from google.colab import files
        print("📂 Upload 'data.zip' (or the files inside the data folder):")
        up = files.upload()
        for name, content in up.items():
            if name.lower().endswith(".zip"):
                zipfile.ZipFile(io.BytesIO(content)).extractall(".")
        DATA = _find_data()
    except ImportError:
        pass
if DATA is None:
    raise FileNotFoundError("No workshop data found. Set DATA_URL above, run from the "
                            "folder that contains 'data/', or upload data.zip on Colab.")

print("✅ Ready. Using data from:", DATA.resolve())

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

# 1 · Where does data come from?

Data doesn't fall from heaven like mana — **someone has to collect it.**

### 1a · Asking an *API*
Some organisations run servers that answer questions over the internet. You send
a web address, they send back data. Let's ask **Open-Meteo** (a free weather
service — no sign-up needed) what the weather is *right now* in Saarbrücken.

In [ ]:
# A web address that asks for the current weather at Saarbrücken (49.23°N, 6.99°E).
url = ("https://api.open-meteo.com/v1/forecast"
       "?latitude=49.23&longitude=6.99"
       "&current=temperature_2m,precipitation,wind_speed_10m"
       "&timezone=Europe/Berlin")

try:
    weather = requests.get(url, timeout=15).json()   # ask, and read the answer
    print("🌍 Fresh data, straight from the internet!\n")
except Exception as e:
    print("(No internet — using a saved copy.)\n")
    weather = json.load(open(DATA / "weather_example.json"))

# The answer comes back as JSON: nested boxes of values. Here it is:
print(json.dumps(weather, indent=2)[:550])

That blob is **JSON** — just boxes inside boxes (Python calls them *dictionaries*
and *lists*). To get one value out, you open the boxes one at a time:
`weather["current"]["temperature_2m"]`.

In [ ]:
# TODO: pull the current temperature out of the `weather` data into `temp`.
# Hint: look at the printout above. There's a "current" box, and inside it a
#       "temperature_2m" box. Open them in order.
temp = ...        # <-- replace the ...

print(f"It is currently {temp}°C in Saarbrücken.")

### 1b · Watching the world (a *scraper*)
Weather is easy — someone built a tidy API for it. But **how busy is a gas
station right now?** Nobody publishes that as a neat feed. It *is* visible, though,
on Google Maps ("Live: busier than usual"). So we wrote a program — a
**scraper** — that automatically checked thousands of places, **every hour, for
seven months**, and saved what it saw.

Here is **one real saved response**: what the scraper recorded for a single gas
station at a single moment.

In [ ]:
poi = json.load(open(DATA / "popular_times_example.json"))

print("Name:            ", poi["title"])
print("What kind:       ", poi["placeTypes"])
print("Captured at:     ", poi["scrapedAtLocal"], poi["timezone"])
print("Busy RIGHT NOW:  ", poi["popularTimesLivePercent"], "%   <-- the key number")

## Pillar 1 · Conceptualization — *define the quantity*
**The one idea: the thing you can measure is never quite the thing you care about.**

What do we *actually* care about? **How much fuel people buy** — litres. But nobody
publishes litres-per-hour. So the researchers reached for the closest thing they
*could* see: how **busy** the gas station is. Swapping the thing you want (litres)
for the thing you can get (busyness) is called using a **proxy**, and it's the first
move in almost every data project.

**busyness is not litres.** Picture
someone who waits for the cheap day and then fills a *bigger* tank — that's **one
visit** (busyness barely moves) but **more litres** (demand jumps). The proxy and
the real thing can drift apart. So every time you see "busyness" today, silently
translate: *our stand-in for fuel demand — close, but not the same thing.*

Google also gives a **"typical week"** — how busy this place *usually* is each
hour. That's the baseline the researchers compare the live number against. Let's
draw the typical **Friday**, with the live reading on top.

In [ ]:
friday = pd.DataFrame(poi["popularTimesHistogram"]["Friday"])

plt.figure()
plt.bar(friday["hour"], friday["occupancyPercent"], color="#cccccc", label="Typical Friday")
plt.axhline(poi["popularTimesLivePercent"], color="#d62728", lw=2.5,
            label=f"Live now: {poi['popularTimesLivePercent']:.0f}%")
plt.xlabel("Hour of day"); plt.ylabel("How busy (%)")
plt.title(poi["title"]); plt.legend(); plt.show()

> ### ⚠️ Bias #1 of 3 · Missing data *(selection at the source)*
> Something sneaky is built into that live number. Google only reports a **live %**
> when a place is **busy enough** to estimate. A sleepy station at 3 a.m.? It just
> returns *nothing* — and "nothing" never lands in our table. So our data is quietly
> tilted toward **busy places and busy times**, and the quiet ones are **invisible**.
>
> This is the nastiest kind of bias: it isn't a wrong number you can spot and fix —
> it's in the rows that **aren't there**. (The paper is honest that it simply *can't*
> measure how often a station goes unreported: those silent moments were dropped
> before the data ever reached the researchers.)

**Pause and appreciate the scale.** That was *one* place at *one* moment. The full
dataset is thousands of places × every hour × seven months ≈ **25 million rows.**
No human could collect that by hand — which is exactly why we automate it. From
here on, we work with the cleaned-up table that all those scrapes became.

# 2 · Plot the V-shape

The 25 million scrapes have already been tidied into a simple table: **one row =
one country, one day, one type of place, and how busy it was on average.** Let's
load it and ask our real question: *did the oil shock change how much people drove?*

In [ ]:
busy = pd.read_csv(DATA / "busyness_daily.csv", parse_dates=["date"])
price = pd.read_csv(DATA / "diesel_price_daily.csv", parse_dates=["date"])

print(busy.shape[0], "rows.  Each row is one country, one day, one place-type.")
busy.head()

In [ ]:
# We start with the 15 countries that DID cut fuel taxes (we'll use the others later).
interveners = busy[busy["intervened"]].copy()
print("Countries that cut taxes:", interveners["country"].nunique())

Now your turn — the payoff cell. We want **one line for gas stations** and **one
for supermarkets**, averaged across all 15 countries, with the **diesel price**
drawn behind them. The plotting is written for you; you just have to build the
daily averages.

In [ ]:
# GOAL: build `daily`, a table with one row per date and two columns: "gas" and "supermarket".
#
# Step 1 — average busyness across countries for each (date, poi_class), then split
#          the two place-types into columns with .unstack("poi_class"):
#   daily = interveners.groupby(["date", "poi_class"])["live_pct"].mean().unstack("poi_class")
#
# Step 2 — smooth the weekday wiggle with a 7-day rolling average:
#   daily = daily.rolling(7, center=True, min_periods=4).mean()

# TODO: write Step 1 and Step 2 (replace the ...)
daily = ...


# ---- everything below is done for you; it runs once `daily` exists ----
fig, ax = plt.subplots()
ax.plot(daily.index, daily["gas"], lw=2.2, color="#1f77b4", label="Gas stations")
ax.plot(daily.index, daily["supermarket"], lw=2.2, color="#2ca02c", label="Supermarkets")
ax.set_ylabel("Live busyness (%)"); ax.legend(loc="lower right")
ax2 = ax.twinx()
ax2.plot(price["date"], price["diesel_price_eur_per_1000l"], "r--", alpha=.6)
ax2.set_ylabel("Diesel price (€ / 1000 L)", color="r")
plt.title("Did the oil shock change how much people drove?")
plt.show()

🎉 **You just reproduced a figure from a real scientific paper.** Read the shape:
as the **diesel price** (red) climbs to its peak around **1 April**, busyness
**dips**; as the price eases, busyness **rebounds**. A clear **V**.

And notice the green line: **supermarkets did the same thing as gas stations.**
So this isn't only about refuelling — people re-timed *all kinds* of car trips.
Keep that in mind; it becomes important soon.

You now have a **result**. Everything from here is the part most people skip and
scientists won't — three hard questions about whether you've *earned* it:
*Can I trust this measurement? What can it actually see? Did the cuts cause it?*

# 3 · Is our measurement even real?

## Pillar 2 · Validation — *trust the quantity*
**The one idea: a pattern you can't independently check is just a pretty picture.**
When your instrument is new, find something you *already* trust — measured a
**completely different way** — and see whether the two agree.

Here's the habit that separates a scientist from someone who just makes charts.
You found a beautiful pattern. **But should you trust it?** "Google busyness" is a
guess Google makes about crowds. What if it's junk, and the V-shape is an artefact?

> *"I found a pattern — but is my measurement even real?"*

The only way to know is to compare it against a **completely independent**
measurement of the same thing. Germany has physical sensors buried in the
motorways (*Autobahn* loop detectors) that literally **count cars**. If our
Google busyness rises and falls on the same days as the real car counts, we can
believe it. Let's load both.

In [ ]:
val = pd.read_csv(DATA / "validation_daily.csv", parse_dates=["date"])
# busyness = our Google signal: avg busyness at German gas stations that day
# flow     = the truth: avg cars-per-hour counted by motorway sensors that day
# speed    = avg car speed on the motorway
# dow      = day of week (0 = Monday ... 6 = Sunday)
val.head()

In [ ]:
# Plumbing — just run this. Weekends are busy everywhere, which could fake an
# agreement between the two signals. We strip out that weekly rhythm from BOTH,
# leaving only the genuine day-to-day ups and downs.
def remove_weekly_rhythm(series, dow):
    return series - dow.map(series.groupby(dow).mean())

val["busyness_clean"] = remove_weekly_rhythm(val["busyness"], val["dow"])
val["flow_clean"]     = remove_weekly_rhythm(val["flow"],     val["dow"])
print("Done — added 'busyness_clean' and 'flow_clean'.")

Now measure the agreement. **Correlation** is one number from −1 to +1: near +1
means "they rise and fall together", near 0 means "no relationship". Your job is
to compute it and *see* it.

In [ ]:
# TODO 1: compute the correlation between the two cleaned signals.
#   Hint: a pandas column has a .corr() method ->  val["A"].corr(val["B"])
r = ...

print(f"Correlation between Google busyness and real car counts:  r = {r:.2f}")

# TODO 2: draw a scatter plot — flow_clean on the x-axis, busyness_clean on the y-axis.
#   Hint: plt.scatter(x_values, y_values, alpha=0.7)
plt.figure()
...        # <-- your scatter plot here
plt.xlabel("Real motorway car counts (weekly rhythm removed)")
plt.ylabel("Google gas-station busyness (weekly rhythm removed)")
plt.title(f"Do two independent measurements agree?   r = {r:.2f}")
plt.show()

You should get **r ≈ 0.45**. Think about what that means: a number Google *guesses*
about crowds, and physical sensors that *count cars on a different part of the
country* — two totally separate systems — **move up and down together.** Our
busyness signal is measuring something real.

That's the most important move in all of data science: **before you trust a
number, check it against something independent.** A pattern you can't validate is
just a pretty picture.

> **🔎 Go deeper (optional).** They agree at r ≈ 0.45 — real, but not 0.95. *Why
> wouldn't two honest measurements of "driving" agree more closely?* Brainstorm three
> reasons. Then flip it: if they'd matched at r ≈ 0.99, would that reassure you — or
> would you start to suspect the two "independent" systems weren't independent at all?

# 4 · What can the data even see?

## Pillar 3 · Resolution — *know the quantity's limits*
**The one idea: data can only answer questions at the scale it was measured.**

We have busyness for every *hour*. What if we'd only had it once a *month* — the way
governments publish official fuel statistics? Let's find out by taking the **exact
same data** and averaging it by month instead of by day.

In [ ]:
# The SAME gas-station data — but averaged by MONTH instead of by day.
g = busy[(busy["poi_class"] == "gas") & busy["intervened"] & (busy["date"] < "2026-06-01")].copy()
order = ["March", "April", "May"]
g["month"] = pd.Categorical(g["date"].dt.month_name(), categories=order, ordered=True)
monthly = g.groupby("month", observed=True)["live_pct"].mean()

plt.figure(figsize=(6, 4))
bars = plt.bar(monthly.index.astype(str), monthly.values, color="#1f77b4")
for b, v in zip(bars, monthly.values):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.4, f"{v:.0f}%", ha="center")
plt.ylabel("Average gas busyness (%)"); plt.title("The SAME data, averaged by MONTH")
plt.show()

**Where did the V go?!** At the daily scale you saw a dramatic plunge to ~22% and a
rebound to ~57%. At the monthly scale you get... a gentle, boring climb: **40% → 48%
→ 50%.** The dip and the rebound happen *inside* a few weeks, so averaging over a
whole month **smears them together until they cancel out.**

This is exactly what happened in the real world. The paper checked the official
**monthly national fuel-sales** numbers (the kind every government publishes). In
March 2026, diesel sales rose **+15.7% in Poland, +13.4% in Italy, +7.8% in Spain** —
all completely **ordinary** for a March. The whole shock-and-rebound left **no
fingerprint** at monthly scale.

So here's the deep point: the researchers didn't just collect *more* data — they
collected it at a **finer resolution**, and that made a *different category of
question* askable. A re-timing of trips within a month is **real**, yet **invisible**
to anyone with only monthly numbers. **Granularity decides what you can even ask.**

> **🔎 Go deeper (optional).** If the dip-and-rebound nets to roughly zero over a
> month, could you *ever* prove it exists using only monthly data? (No.) So: is a
> claim you can never check with the data on hand a *finding*, or a kind of *faith*?
> When is "you'd need finer data to see it" a fair scientific statement — and when is
> it a convenient way to dodge being wrong?

# 5 · The trap: did the tax cuts cause it?

## Pillar 4 · Causal attribution — *constrain what you conclude*
**The one idea: a change that comes *after* an event is not proof the event caused
it. The hardest, most honest move is refusing to claim what your data can't support.**

Sixteen governments cut fuel taxes during exactly these weeks. The obvious story
writes itself: *taxes cut → fuel cheaper → people drive more → the rebound.* Let's
check the "naive" version a journalist might run.

In [ ]:
gas = busy[busy["poi_class"] == "gas"]
before = gas[gas["date"] < "2026-03-20"].groupby("intervened")["live_pct"].mean()
after  = gas[gas["date"] > "2026-05-01"].groupby("intervened")["live_pct"].mean()

print(f"Tax-cutting countries were  {before[True]:.0f}% busy BEFORE the cuts")
print(f"                    ... and {after[True]:.0f}% busy AFTER  the cuts")
print(f"\n👉 A jump of +{after[True]-before[True]:.0f} points. 'The cuts worked!'")

> ### ⚠️ Bias #2 of 3 · A bad reference point *(contaminated baseline)*
> We just leaned on "before the cuts" vs "after." But look hard at our **before**:
> it's early March 2026 — and the war, and the price spike, had *already started* on
> 28 February. So our "before" isn't a calm, normal baseline at all — it's **already
> inside the storm.** When the starting line has already moved, every change you
> measure *from* it is suspect. A truly clean "before" would need data from *before
> the war* — which, for most countries, the researchers simply don't have.

A reporter could now write: *"After Europe slashed fuel taxes, station traffic
surged. The policy worked."* The actual researchers ran a careful statistical model
and got the same flavour of answer: **+6.4 points, p < 0.001** — "highly significant."

Pause on that word. **Significant** means "probably not random luck." It does **not**
mean "real effect," and it certainly doesn't mean "caused by the cuts." A number can
be measured precisely, *be* statistically significant, **and still be an artefact.**
Don't take my word for it — here's the proof. Let's look at three countries that cut
**no tax at all**: France, the United Kingdom, the Netherlands. If the *cuts* caused
the rebound, these should look **different**.

In [ ]:
noncut = ["France", "United Kingdom", "Netherlands"]

# TODO: for each country in `noncut`, plot its daily GAS-station busyness over time
#       (7-day smoothed, so it's readable).
#   Hint: loop over noncut. For each country, take the rows where
#         busy["country"] == country AND busy["poi_class"] == "gas",
#         sort by date, smooth with .rolling(7, center=True, min_periods=4).mean(),
#         and plt.plot(dates, values, label=country).
plt.figure()
for country in noncut:
    ...     # filter -> smooth -> plot this country's gas busyness

plt.ylabel("Gas-station busyness (%)")
plt.title("Three countries that cut NO fuel tax")
plt.legend(); plt.show()

**The exact same V-shape.** France, the UK, and the Netherlands cut nothing, yet
their driving dipped and rebounded *just like* the cutting countries. The "+6.4-point
policy effect" was real, and significant — and an **artefact**. The rebound wasn't
the cuts; it was the shared shock and the Easter holidays hitting *everyone* at once.

> ### ⚠️ Bias #3 of 3 · Tangled causes *(confounding)*
> Why couldn't the careful model save us? Because **three things moved in lockstep**,
> in the same six weeks: the **price** spiked and fell, **Easter** holidays came and
> went, and the **tax cuts** landed. When your suspect (the cuts) always shows up at
> the exact moment as the other suspects (price, holidays), no amount of clever maths
> can say which one did it. They're **confounded** — tangled together — and the data
> cannot pull them apart. This is the bias living inside Pillar 4.

---
### 🗣️ The debate — this *is* the workshop (≈20 min, 2 vs 2)
Be rigorous, not polite. Pick sides.

**Pair A — "the cuts worked."** Your evidence is *the number*: a careful model says
the cutting countries' busyness rose **+6.4 points after the cut, p < 0.001** (under
1-in-1000 odds it's a fluke). Build the strongest possible case that this is a real
policy effect — make it genuinely hard to beat.

**Pair B — "you can't claim that."** Explain how a number that solid can still be an
artefact. Your weapon is the three no-cut countries. And do **not** say the cuts
*failed* — say something more precise.

**Then, all four of you, agree on ONE sentence.** You must choose exactly one:
- **(a)** "The tax cuts had *no effect* on driving."
- **(b)** "We *cannot tell* whether the tax cuts had an effect."

These are **not** the same sentence, and the gap between them is the whole point of
the day. Which one does the evidence actually support — and why is choosing the wrong
one its own kind of dishonesty?

### 🧭 Last question — turn "we can't tell" into a plan
"We can't tell" sounds like a dead end. It's the opposite — it's a **blueprint for a
better study.** What would the world have had to look like for you to *actually*
isolate the cuts' effect? Push on three things:

1. **Timing** — what if the cuts had been spread across the whole year, instead of all
   jammed into the same six weeks as the price spike and Easter?
2. **Controls** — what if some countries had credibly promised to *never* cut, giving
   you a clean comparison group that nothing else disturbed?
3. **Baseline** — what if you had real data from *before the war* — a genuinely calm
   "before"?

Name what you'd need. (That's literally how good studies get designed: by working out
exactly what the messy real world refused to hand you.)

# 🏁 What you actually learned

You ran a real research project — but the four moves you made are far bigger than fuel:

1. **Conceptualization** — you measured *busyness* as a stand-in for *fuel demand*,
   and never forgot they aren't the same thing.
2. **Validation** — you refused to trust the signal until it agreed with a totally
   independent one (the motorway sensors, r ≈ 0.45).
3. **Resolution** — you saw the whole phenomenon vanish at monthly scale, so the
   question was only *askable* because the data was fine-grained.
4. **Causal attribution** — you found a real, significant, **meaningless** number,
   and had the discipline to say *"we cannot tell"* instead of *"it worked."*

And you met **bias** in all three of its disguises: data that's **missing** (Google's
silent quiet places), a **reference point that already moved** (a "before" inside the
war), and **causes that are tangled** (price, Easter, and cuts in lockstep).

The most grown-up sentence in all of science is the one you practised today:
**"we cannot tell" is not the same as "there is no effect."** Knowing the difference
— and being brave enough to say the first one out loud — is the whole job.